# VRI 2026 - YOLO Platform & Markers Tracker

This notebook unzips the `colab_dataset.zip` from your Google Drive, installs Ultralytics, and trains the YOLOv8-Pose model on the combined manual and synthetic datasets.

In [ ]:
# Mount Google Drive so we can access the dataset and save the .pt weights
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the dataset and install Ultralytics
!unzip -q -o /content/drive/MyDrive/colab_dataset.zip -d /content/
!pip install ultralytics
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Train the model natively in Python
from ultralytics import YOLO
import os

yaml_path = '/content/dataset/colab_dataset.yaml'
if not os.path.exists(yaml_path):
    print(f"ERROR: Could not find {yaml_path}. Make sure the dataset was unzipped correctly!")
else:
    model = YOLO('yolov8n-pose.pt')
    
    project_dir = '/content/drive/MyDrive/YOLO_Models'
    os.makedirs(project_dir, exist_ok=True)
    
    results = model.train(
        data=yaml_path,
        epochs=300,            # Increased from 100. YOLO has early-stopping by default (patience=50), so it will stop automatically when it peaks.
        imgsz=640,
        batch=32,              # Colab T4 GPUs have 16GB VRAM, so batch=32 will run faster and provide smoother gradient updates.
        project=project_dir,
        name='yolov8_platform_pose_markers_v4',
        exist_ok=True,
        
        # --- Loss Weights ---
        pose=15.0,             # (Default 12.0) Increase pose loss weight to force the model to prioritize keypoint pixel accuracy over bounding box accuracy.
        kobj=2.0,              # (Default 1.0) Increase keypoint objectness penalty to prevent it from hallucinating keypoints where there are none.
        box=7.5,               # Keep default box loss.

        # --- Augmentations ---
        # Since our camera is mostly fixed top-down, extreme scaling isn't necessary, but small variations help robustness.
        perspective=0.0005,    # Reduced slightly from 0.001. We want *some* perspective distortion for robustness against camera bumps, but not extreme warps.
        degrees=10.0,          # Reduced from 15.0. 
        scale=0.2,             # Reduced from 0.5. Camera height is fixed, so extreme zooming out/in isn't needed.
        mosaic=1.0,            # Keep at 1.0. Excellent for small datasets.
        
        # --- Colors ---
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4
    )

    print(f"Training complete! Model saved in {project_dir}/platform_and_markers_model/weights/best.pt")
